---


## Task 2: Recursive utility functions

For task 2, we are writing two recursive helper functions that are used in the full workflow in task 5

1. `find_log_files` - searches through the nested `d2_data/`folder tructure (year → store → register) and returns all `.log`files. 
2. `find_active_registers`- traverses the nested `store_config.json` file and returns the IDs of all registers marked as active.

Both fuctions use the explicit recursion: each call handles one level of the structure and calls itself on the levels below. As required, we do not use `Path.glob()`, `Path.rglob()` or `os.walk()`.


### Task 2.1: Recursive search for log files 

The function takes a folder path and converts it to a `pathlib.Path` object. It then loops everything directly inside the folder using `Path.iterdir()`: 

* If the item is a **folder**, the funcion calls itself on that folder and adds the files it ends up finding to the list (this is the recursive step).
* If the item is a **file** with the right expression, it is added to the list. 
* Other files (e.g `store_config.json`) are ignored 

The recursion stops when a folder only contains files, since no new calls are made, which is the best case. The list is sorted and returned, so the order of files is the same every time the code runs.

The input path has the type hints `str | Path`, since the function accept both a string and a `Path` object. 




In [4]:
def find_log_files(path: str | Path, extension: str = ".log") -> list[Path]:
    """
    Recursivly find all files with a given extension in a dictionay tree.    ← What the function does
    
    Paramaters                                                               ← the inputs 
    ----------
    path : str | Path
        The directory to serach in. COnverted to a pathlib.Path object.
    extension : str, optional
        The file extension to searh for, including the dot (default ".log").
        
    Returns                                                                  ← What comes back                                         
    -------
    list[Path]
        Sorted list of path objects for all matching files in the directory 
        and all of its subdirectories.
    """
    path = Path(path)
    found: list[Path] = []

    for item in path.iterdir():
        if item.is_dir():
            #Recursive step: Search inside the subdirectory 
            found.extedn(find_log_files(item, extension))
        elif item.is_file() and item-suffix == extension: 
            found.append(item)

        return sorted(found) 

---

### 2.2 Recursive search for active registers

The configuration file contains dictionaries and lists nested inside each other. Since we don't want the function to depend on the exact structure, it handles any combination of the two:

* If the input is a **dictionary**, we check whether it has `"active": True` and an `"id"`. If so, the id is added to the set. The function then calls itself on every value in the dictionary.
* If the input is a **list**, the function calls itself on every element.
* Anything else (strings, numbers, booleans, `None`) returns an empty set. This is the base case.

The results from each recursive call are combined using set union (`|=`). We return a `set[str]` so the same register id is never stored twice. Note that `true` in JSON becomes `True` in Python when the file is loaded with `json.load`.

To give the input a precise type hint, we define the type alias `JSONValue`, which describes every type a value loaded from a JSON file can have. The alias refers to itself, since a dictionary or list can contain more dictionaries and lists.

In [5]:
JSONValue = dict[str, "JSONValue"] | list["JSONValue"] | str | int | float | bool | None 

def find_active_registers(node: JSONValue) -> set[str]:
    """
    Recursivly collect the ids of all active registers in a nested JSON structure.   ← What the function does
    
    Paramaters                                                                       ← the inputs 
    ----------
    node: JSONValue
        Any part of the loaded JSON data: a dictionary, a list or a single value. 
        
    Returns                                                                          ← What comes back   
    -------
    set[str]
        The ids of all registers "active" is True. Returns an empty set 
        if node is not a dictionary or a list (the base case).
    """
    active: set[str] = set()

    if isinstance(node, dict):
        if node.get("active") is True and "id" in node:
            active.add(str(node["id"]))
        # Recursive step: Search evey value in the dictionary 
        for value in node-values():
            active |= find_active_registers(value)

    elif isinstance(node, list):
        # Recursive step: Search every element in the list 
        for element in node:
            active |= find_active_registers(element)

    return active 